# Task 1 — Thematic Factor Discovery: Sentiment Scoring

Covers **Task 1, Sub-step 3**: use the LLM to assign 5-class sentiment labels to every factor extracted in Sub-step 2.

For each filing's factor JSON, the LLM scores factors in batches of 8, assigning a label, rationale, and confidence.

| | |
|---|---|
| **Input** | `output/factors/{TICKER}/{STEM}_factors.json` |
| **Output** | `output/factors_scored/{TICKER}/{STEM}_factors.json` |
| **Model** | `Qwen/Qwen2.5-14B-Instruct` via vLLM on ACCRE |
| **GPU** | Required — RTX A6000, account `p_dsi_acc` |

| Label | Meaning |
|---|---|
| `very_negative` | Clear, significant deterioration or major risk |
| `negative` | Moderate headwind, declining trend, or concern |
| `neutral` | Stable, mixed signals, or no clear directional signal |
| `positive` | Moderate improvement, tailwind, or opportunity |
| `very_positive` | Clear, significant improvement or major positive development |

## Step 1: Imports & Configuration

Set up paths, logging, and LLM constants. The vLLM server must be running before executing Step 3+.

In [ ]:
import json
import logging
import sys
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Optional

from openai import OpenAI
from tqdm.notebook import tqdm

# ── Paths ────────────────────────────────────────────────────────────────
_cwd = Path.cwd()

_search = _cwd
FILLINGS_ROOT = None
for _ in range(6):
    if (_search / "Data").is_dir():
        FILLINGS_ROOT = _search
        break
    _search = _search.parent

if FILLINGS_ROOT is None:
    raise FileNotFoundError(
        f"Could not find Data/ directory in any parent of {_cwd}. "
        "Make sure the notebook is inside the fillings/ project tree."
    )

FACTORS_DIR = _cwd / "output" / "factors"
FACTORS_SCORED_DIR = _cwd / "output" / "factors_scored"
TICKER_MAPPING_PATH = _cwd / "ticker_mapping.json"

FACTORS_SCORED_DIR.mkdir(parents=True, exist_ok=True)

# ── LLM Constants ────────────────────────────────────────────────────────
BASE_URL = "http://127.0.0.1:8000/v1"
API_KEY = "local"
MODEL = "Qwen/Qwen2.5-14B-Instruct"
TEMPERATURE = 0.0
MAX_RETRIES = 5
RETRY_BASE_DELAY = 2.0
REQUEST_TIMEOUT = 300
MAX_IN_FLIGHT_LLM = 4

# ── Scoring Constants ─────────────────────────────────────────────────────
BATCH_SIZE = 8  # factors per LLM call

# ── Logging ──────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
log = logging.getLogger("task_1_3")

print(f"Fillings root:        {FILLINGS_ROOT}")
print(f"Factors dir (input):  {FACTORS_DIR} (exists={FACTORS_DIR.exists()})")
print(f"Scored dir (output):  {FACTORS_SCORED_DIR}")
print(f"Ticker mapping:       {TICKER_MAPPING_PATH} (exists={TICKER_MAPPING_PATH.exists()})")
print(f"Model:                {MODEL}")
print(f"Batch size:           {BATCH_SIZE} factors per LLM call")
print(f"Concurrency:          {MAX_IN_FLIGHT_LLM} in-flight LLM calls")

## Step 2: LLM Client Utilities

OpenAI-compatible client with:
- **Concurrency control** — `BoundedSemaphore` limits concurrent LLM calls
- **Exponential backoff** — retries on transient failures
- **JSON parsing** — strips markdown code blocks, extracts JSON from mixed output

In [ ]:
# Global concurrency limiter
_semaphore = threading.BoundedSemaphore(MAX_IN_FLIGHT_LLM)


def call_llm(
    client: OpenAI,
    system_prompt: str,
    user_prompt: str,
    temperature: float = TEMPERATURE,
    max_tokens: int = 4096,
) -> Optional[str]:
    """
    Call the LLM with retry logic and concurrency control.
    Returns the response content string, or None on failure.
    """
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with _semaphore:
                response = client.chat.completions.create(
                    model=MODEL,
                    messages=[
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    temperature=temperature,
                    max_tokens=max_tokens,
                    timeout=REQUEST_TIMEOUT,
                )
            return response.choices[0].message.content

        except Exception as e:
            delay = RETRY_BASE_DELAY * (2 ** (attempt - 1))
            log.warning(
                f"  LLM call attempt {attempt}/{MAX_RETRIES} failed: {e}. "
                f"Retrying in {delay}s..."
            )
            if attempt < MAX_RETRIES:
                time.sleep(delay)
            else:
                log.error(f"  LLM call failed after {MAX_RETRIES} attempts: {e}")
                return None


def parse_json_response(content: str) -> Optional[Any]:
    """Parse JSON from LLM response, handling markdown code blocks."""
    content = content.strip()

    # Strip markdown code blocks if present
    if content.startswith("```"):
        content = content.split("\n", 1)[1] if "\n" in content else content[3:]
        if content.endswith("```"):
            content = content[:-3]
        content = content.strip()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        pass

    # Fallback: find the first { or [ and last } or ]
    start_obj = content.find("{")
    start_arr = content.find("[")
    if start_obj == -1 and start_arr == -1:
        log.error(f"  No JSON found in response: {content[:200]}...")
        return None

    if start_arr != -1 and (start_obj == -1 or start_arr < start_obj):
        start = start_arr
        end = content.rfind("]")
    else:
        start = start_obj
        end = content.rfind("}")

    if end == -1 or end <= start:
        log.error(f"  Malformed JSON in response: {content[:200]}...")
        return None

    try:
        return json.loads(content[start:end + 1])
    except json.JSONDecodeError as e:
        log.error(f"  JSON parse error: {e}. Content: {content[start:start+200]}...")
        return None


def call_llm_json(
    client: OpenAI,
    system_prompt: str,
    user_prompt: str,
    temperature: float = TEMPERATURE,
    max_tokens: int = 4096,
) -> Optional[Any]:
    """Call the LLM and parse the response as JSON."""
    content = call_llm(client, system_prompt, user_prompt, temperature, max_tokens)
    if content is None:
        return None
    return parse_json_response(content)


# ── Create client & health check ─────────────────────────────────────────
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

try:
    models = client.models.list()
    available = [m.id for m in models.data]
    print(f"vLLM is reachable. Available models: {available}")
    if MODEL not in available:
        print(f"WARNING: {MODEL} not in available models!")
except Exception as e:
    print(f"WARNING: vLLM health check failed: {e}")
    print("The LLM server must be running before executing Steps 3+.")

## Step 3: Sentiment Scoring Logic

Each batch of factors is sent to the LLM with their summary and evidence.
The LLM returns a 5-class sentiment label, a 1-sentence rationale, and a confidence score.

- **`build_sentiment_prompt()`** — formats factors into a structured prompt
- **`score_batch()`** — calls the LLM, parses and validates the response

In [ ]:
SENTIMENT_SYSTEM_PROMPT = """\
You are a financial analyst assigning sentiment labels to thematic factors extracted from SEC filings.

For each factor, you will receive its summary and evidence quotes from the filing.

Assign a sentiment label from exactly these five classes:
- "very_negative": Clear, significant deterioration or major risk
- "negative": Moderate headwind, declining trend, or concern
- "neutral": Stable, mixed signals, or no clear directional signal
- "positive": Moderate improvement, tailwind, or opportunity
- "very_positive": Clear, significant improvement or major positive development

Respond with a JSON array where each entry has:
{
  "key": "<factor_key>",
  "label": "<one of: very_negative, negative, neutral, positive, very_positive>",
  "rationale": "<1 sentence explaining why you chose this label>",
  "confidence": <float 0.0 to 1.0>
}

Rules:
- Base your label ONLY on the summary and evidence provided
- Be decisive — avoid defaulting to "neutral" unless genuinely mixed
- Confidence should reflect how clearly the text supports a directional reading
- Return valid JSON array only, no other text"""

VALID_LABELS = {"very_negative", "negative", "neutral", "positive", "very_positive"}


def build_sentiment_prompt(factors_batch: list[dict]) -> str:
    """Build user prompt for a batch of factors."""
    blocks = []
    for factor in factors_batch:
        evidence_text = "\n".join(
            f"  - \"{e['text']}\""
            for e in factor.get("evidence", [])
            if isinstance(e, dict) and e.get("text")
        )
        blocks.append(
            f"### Factor: {factor['key']} (category: {factor['category']})\n"
            f"Summary: {factor['summary']}\n"
            f"Evidence:\n{evidence_text}"
        )

    return (
        "\n\n".join(blocks)
        + "\n\nAssign sentiment labels to each factor above. Return a JSON array."
    )


def score_batch(
    client: OpenAI, factors_batch: list[dict]
) -> dict[str, dict]:
    """
    Score a batch of factors for sentiment.

    Returns {factor_key: {label, rationale, confidence}} for successfully scored factors.
    """
    prompt = build_sentiment_prompt(factors_batch)
    result = call_llm_json(client, SENTIMENT_SYSTEM_PROMPT, prompt, max_tokens=2048)

    if result is None or not isinstance(result, list):
        log.warning(
            f"    Sentiment LLM returned invalid result for batch of {len(factors_batch)}"
        )
        return {}

    scored: dict[str, dict] = {}
    for item in result:
        if not isinstance(item, dict):
            continue
        key = item.get("key")
        label = item.get("label")
        if not key or label not in VALID_LABELS:
            continue
        scored[key] = {
            "label": label,
            "rationale": item.get("rationale", ""),
            "confidence": min(1.0, max(0.0, float(item.get("confidence", 0.5)))),
        }

    return scored


print("Sentiment scoring functions loaded.")
print(f"Valid labels: {sorted(VALID_LABELS)}")
print(f"Batch size: {BATCH_SIZE} factors per LLM call")

## Step 4: Filing-Level Processing

Loads one factor JSON, batches its factors into groups of `BATCH_SIZE`, and scores all batches
**concurrently** via `ThreadPoolExecutor` (same pattern as task_1_2).

Each filing has ~30 factors → ~4 batches of 8 → all sent in parallel → completes in one round-trip (~10s) instead of four (~40s).

In [ ]:
def process_filing(factor_file: Path, client: OpenAI) -> dict | None:
    """
    Score all factors in one filing for sentiment.

    Batches are scored concurrently via ThreadPoolExecutor to leverage
    vLLM's continuous batching on the GPU.

    Returns the updated factor JSON with sentiment populated, or None on failure.
    """
    with open(factor_file) as f:
        data = json.load(f)

    factors = data.get("factors", [])
    if not factors:
        return None

    # Build batches
    batches = [
        factors[i : i + BATCH_SIZE]
        for i in range(0, len(factors), BATCH_SIZE)
    ]

    # Score all batches concurrently
    all_scored: dict[str, dict] = {}

    with ThreadPoolExecutor(max_workers=MAX_IN_FLIGHT_LLM) as pool:
        futures = {
            pool.submit(score_batch, client, batch): i
            for i, batch in enumerate(batches)
        }
        for future in as_completed(futures):
            batch_idx = futures[future]
            try:
                scored = future.result()
                all_scored.update(scored)
            except Exception as e:
                log.warning(f"    Batch {batch_idx} failed: {e}")

    # Apply sentiment labels to factors
    scored_count = 0
    for factor in factors:
        sentiment = all_scored.get(factor["key"])
        if sentiment:
            factor["sentiment"] = sentiment
            scored_count += 1
        else:
            factor["sentiment"] = None

    log.info(
        f"  {data['ticker']} {data['form']} {data['filing_date']}: "
        f"{scored_count}/{len(factors)} factors scored"
    )

    return data


print("Filing-level processing function loaded.")

## Step 5: Filing Discovery

Load tickers, discover factor files, compute output paths, and check resume state.
Prints a dry-run summary before actual processing.

In [ ]:
# Load tickers
with open(TICKER_MAPPING_PATH) as f:
    ticker_mapping = json.load(f)

all_tickers = sorted(
    t for sector_list in ticker_mapping.values() for t in sector_list
)
print(f"Loaded {len(all_tickers)} tickers across {len(ticker_mapping)} sub-sectors")

# ── Filter to AAL only for testing (comment out to run all tickers) ──────
all_tickers = ["AAL"]
print(f"FILTERED to {all_tickers} for testing")

# ── Dry-run summary ───────────────────────────────────────────────────────
total_files = 0
total_already = 0
total_remaining = 0
tickers_with_data = 0
tickers_no_data = []

for ticker in all_tickers:
    ticker_input = FACTORS_DIR / ticker
    if not ticker_input.exists():
        tickers_no_data.append(ticker)
        continue

    factor_files = sorted(ticker_input.glob("*_factors.json"))
    if not factor_files:
        tickers_no_data.append(ticker)
        continue

    tickers_with_data += 1
    ticker_output = FACTORS_SCORED_DIR / ticker

    for ff in factor_files:
        total_files += 1
        out_file = ticker_output / ff.name
        if out_file.exists():
            total_already += 1
        else:
            total_remaining += 1

print(f"\n{tickers_with_data}/{len(all_tickers)} tickers have factor files")
print(f"{total_files} total factor files")
print(f"{total_already} already scored (will be skipped)")
print(f"{total_remaining} remaining to score")

if tickers_no_data:
    print(
        f"\nTickers with no factor data ({len(tickers_no_data)}): "
        f"{', '.join(tickers_no_data)}"
    )

## Step 6: Run Sentiment Scoring

Main processing loop. For each filing:
1. Skip if output already exists (resume-safe)
2. Call `process_filing()` to score all factors
3. Write scored factor JSON to `output/factors_scored/`

In [ ]:
# Build flat list of (ticker, factor_file) pairs
all_jobs: list[tuple[str, Path]] = []
for ticker in all_tickers:
    ticker_input = FACTORS_DIR / ticker
    if not ticker_input.exists():
        continue
    for ff in sorted(ticker_input.glob("*_factors.json")):
        all_jobs.append((ticker, ff))

print(f"{len(all_jobs)} filing jobs queued")

g_ok = g_skip = g_err = g_scored = 0

pbar = tqdm(all_jobs, desc="Sentiment scoring", unit="filing")
for ticker, factor_file in pbar:
    ticker_output = FACTORS_SCORED_DIR / ticker
    ticker_output.mkdir(parents=True, exist_ok=True)

    out_file = ticker_output / factor_file.name

    # Resume-safe
    if out_file.exists():
        g_skip += 1
        pbar.set_postfix(ok=g_ok, skip=g_skip, err=g_err, scored=g_scored)
        continue

    try:
        result = process_filing(factor_file, client)
        if result:
            with open(out_file, "w") as f:
                json.dump(result, f, indent=2)
            g_ok += 1
            g_scored += sum(
                1 for fac in result["factors"]
                if fac.get("sentiment") is not None
            )
        else:
            g_err += 1
    except Exception as e:
        g_err += 1
        log.error(f"  {factor_file.name}: CRASHED -- {e}")

    pbar.set_postfix(ok=g_ok, skip=g_skip, err=g_err, scored=g_scored)

print(f"\nDone. {g_ok} processed, {g_skip} skipped, {g_err} errors")
print(f"Total factors scored: {g_scored}")
if g_ok > 0:
    print(f"Avg factors/filing: {g_scored / g_ok:.1f}")

## Step 7: Scoring Summary

Aggregate stats: total files, factors scored, per-ticker breakdown,
label distribution, average confidence, and a sample factor with its sentiment.

In [ ]:
from collections import Counter

# ── Aggregate stats ───────────────────────────────────────────────────────
ticker_stats = []
total_scored_files = 0
total_factors_all = 0
total_factors_with_sentiment = 0
label_counts: Counter = Counter()
all_confidences: list[float] = []

for ticker in all_tickers:
    ticker_dir = FACTORS_SCORED_DIR / ticker
    if not ticker_dir.exists():
        continue

    scored_files = sorted(ticker_dir.glob("*_factors.json"))
    n_files = len(scored_files)
    total_scored_files += n_files

    n_factors = 0
    n_scored = 0
    for sf in scored_files:
        try:
            with open(sf) as f:
                data = json.load(f)
            for fac in data.get("factors", []):
                n_factors += 1
                sentiment = fac.get("sentiment")
                if isinstance(sentiment, dict) and sentiment.get("label"):
                    n_scored += 1
                    label_counts[sentiment["label"]] += 1
                    if "confidence" in sentiment:
                        all_confidences.append(sentiment["confidence"])
        except Exception:
            pass

    total_factors_all += n_factors
    total_factors_with_sentiment += n_scored
    ticker_stats.append(
        {"ticker": ticker, "files": n_files, "factors": n_factors, "scored": n_scored}
    )

print("=" * 60)
print("SENTIMENT SCORING SUMMARY")
print(f"  Total tickers with output: {len(ticker_stats)}")
print(f"  Total scored files: {total_scored_files}")
print(f"  Total factors: {total_factors_all}")
print(f"  Factors with sentiment: {total_factors_with_sentiment}")
if total_factors_all > 0:
    pct = total_factors_with_sentiment / total_factors_all * 100
    print(f"  Scoring rate: {pct:.1f}%")

# ── Label distribution ─────────────────────────────────────────────────────
print(f"\nLabel Distribution:")
label_order = ["very_negative", "negative", "neutral", "positive", "very_positive"]
for label in label_order:
    count = label_counts.get(label, 0)
    if total_factors_with_sentiment > 0:
        pct = count / total_factors_with_sentiment * 100
        bar = "#" * int(pct / 2)
        print(f"  {label:16s}: {count:5d} ({pct:5.1f}%) {bar}")
    else:
        print(f"  {label:16s}: {count:5d}")

# ── Avg confidence ─────────────────────────────────────────────────────────
if all_confidences:
    avg_conf = sum(all_confidences) / len(all_confidences)
    print(f"\nAvg confidence: {avg_conf:.3f}")

# ── Per-ticker breakdown ──────────────────────────────────────────────────
print(f"\nPer-ticker breakdown:")
for s in sorted(ticker_stats, key=lambda x: x["files"], reverse=True)[:20]:
    print(
        f"  {s['ticker']:6s}: {s['files']:3d} files, "
        f"{s['scored']:4d}/{s['factors']:4d} factors scored"
    )
if len(ticker_stats) > 20:
    print(f"  ... and {len(ticker_stats) - 20} more tickers")

# ── Sample output ─────────────────────────────────────────────────────────
sample_file = None
for ticker in all_tickers:
    ticker_dir = FACTORS_SCORED_DIR / ticker
    if ticker_dir.exists():
        files = sorted(ticker_dir.glob("*_factors.json"))
        if files:
            sample_file = files[0]
            break

if sample_file:
    with open(sample_file) as f:
        sample = json.load(f)
    print(f"\n{'=' * 60}")
    print(f"SAMPLE: {sample_file.name}")
    print(
        f"  Ticker: {sample['ticker']}, Form: {sample['form']}, "
        f"Date: {sample['filing_date']}"
    )
    print(f"  Factors: {sample['num_factors']}")
    for fac in sample["factors"][:3]:
        print(f"\n  [{fac['category']}] {fac['key']}")
        print(f"    Summary: {fac['summary'][:120]}...")
        sentiment = fac.get("sentiment")
        if isinstance(sentiment, dict):
            print(f"    Sentiment: {sentiment.get('label')}")
            print(f"    Rationale: {sentiment.get('rationale', '')}")
            print(f"    Confidence: {sentiment.get('confidence', '')}")
        else:
            print(f"    Sentiment: {sentiment}")
    if sample["num_factors"] > 3:
        print(f"\n  ... and {sample['num_factors'] - 3} more factors")
else:
    print("\nNo scored factor files found yet. Run Step 6 first.")